In [6]:
import torch

In [7]:
#choose cpu/gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [8]:
#load Sigma-Sampler
%run Sigma-Sampler.ipynb

In [10]:
#loss function
def EDM_loss(model, x):
    #reshape the distribution coefficient of noise
    batch_size = x.shape[0]
    sigma = sigma_sampler_lognormal(batch_size).to(device)
    sigma_reshaped = sigma.view(-1, 1, 1, 1)
    #calculate noise
    noise = torch.randn_like(x)
    #calculate img with noise
    x_noisy = x + sigma_reshaped * noise
    #EDM preconditioning
    sigma_data = 0.5
    #scaler factor - input image
    c_skip = sigma_data ** 2 / (sigma_reshaped ** 2 + sigma_data ** 2)
    c_out = sigma_reshaped * sigma_data / torch.sqrt(sigma_reshaped ** 2 + sigma_data ** 2)
    c_in = 1 / torch.sqrt(sigma_reshaped ** 2 + sigma_data ** 2)
    #calculate predictive img
    pred = model(c_in * x_noisy, sigma)
    #x0 target branch(original EDM)
    #construct D_x
    D_x = c_skip * x_noisy + c_out * pred
    #standard weight
    w_edm = (sigma_reshaped ** 2 + sigma_data ** 2) / (sigma_reshaped * sigma_data) ** 2
    #calculate loss value
    loss = w_edm * (D_x - x) ** 2
    return loss.mean(), sigma